# Pharmacovigilance Reaction Categorisation

This notebook develops a transparent approach to categorising
adverse-event reaction terms from openFDA reports.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sys
import importlib

module_folder = "/content/drive/MyDrive/PV_project"

if module_folder not in sys.path:
    sys.path.insert(0, module_folder)

import pv_analysis
importlib.reload(pv_analysis)

print("Module loaded from:", pv_analysis.__file__)
print("Review function available:",
      hasattr(pv_analysis, "flag_reports_for_review"))

Module loaded from: /content/drive/MyDrive/PV_project/pv_analysis.py
Review function available: True


In [ ]:
import pandas as pd

file_path = "/content/drive/MyDrive/PV_project/pv_reports.csv"

df = pd.read_csv(file_path)

print("Reports loaded:", len(df))
print("DataFrame shape:", df.shape)
print(df.head())

Reports loaded: 900
DataFrame shape: (900, 10)
   report_id  receivedate  serious patient_sex reporter_country  \
0  5801206-7     20080707        1           M           CANADA   
1   10003300     20140306        1           F               US   
2   10003301     20140228        1           F               US   
3   10003302     20140312        2           M               US   
4   10003304     20140312        2           F               US   

                              reaction_terms  \
0        DRUG ADMINISTRATION ERROR; OVERDOSE   
1  Vomiting; Diarrhoea; Arthralgia; Headache   
2                Dyspepsia; Renal impairment   
3                           Drug ineffective   
4                      Drug hypersensitivity   

                                          drug_names  \
0                                      DURAGESIC-100   
1                                             BONIVA   
2                                          IBUPROFEN   
3                                    

In [ ]:
reaction_df = (
    df.assign(
        reaction_term=df["reaction_terms"].fillna("").str.split("; ")
    )
    .explode("reaction_term")
)

reaction_df = reaction_df.loc[
    reaction_df["reaction_term"].ne(""),
    ["report_id", "serious", "reaction_term"]
].reset_index(drop=True)

print("Reaction records:", len(reaction_df))
print("Unique reaction terms:", reaction_df["reaction_term"].nunique())

Reaction records: 2460
Unique reaction terms: 855


In [ ]:
pv_review_df = pv_analysis.flag_reports_for_review(df)

print("Reports available:", len(pv_review_df))
print("Reports flagged for review:",
      pv_review_df["review_required"].sum())

Reports available: 900
Reports flagged for review: 130


In [ ]:
for name in ["df", "reaction_df", "pv_review_df"]:
    print(name, "available:", name in globals())

df available: True
reaction_df available: True
pv_review_df available: True


In [ ]:
reaction_df["reaction_term"].drop_duplicates().head(30).tolist()

['DRUG ADMINISTRATION ERROR',
 'OVERDOSE',
 'Vomiting',
 'Diarrhoea',
 'Arthralgia',
 'Headache',
 'Dyspepsia',
 'Renal impairment',
 'Drug ineffective',
 'Drug hypersensitivity',
 'Cough',
 'Throat irritation',
 'Rhinorrhoea',
 'Unevaluable event',
 'Malaise',
 'Hepatic enzyme increased',
 'Back pain',
 'Oedema peripheral',
 'Fluid retention',
 'Pneumonia',
 'Swelling',
 'Jaundice',
 'Death',
 'Oedema',
 'Fatigue',
 'Dyspnoea exertional',
 'Angioedema',
 'Laryngeal oedema',
 'Face oedema',
 'Tuberculosis of central nervous system']

In [ ]:
def categorise_reaction_terms(reaction_df):
    """Apply transparent, rule-based reaction term categories."""

    reaction_df = reaction_df.copy()

    category_mapping = {
        "DRUG ADMINISTRATION ERROR": "Medication use / administration",
        "OVERDOSE": "Medication use / administration",
        "DRUG INEFFECTIVE": "Treatment effectiveness",
        "DEATH": "Patient outcomes",
        "DRUG HYPERSENSITIVITY": "Hypersensitivity / allergic reactions",
        "ANGIOEDEMA": "Hypersensitivity / allergic reactions",
    }

    reaction_df["reaction_category"] = (
        reaction_df["reaction_term"]
        .str.upper()
        .map(category_mapping)
        .fillna("Other clinical events")
    )

    return reaction_df

In [ ]:
import importlib
importlib.reload(pv_analysis)

print(hasattr(pv_analysis, "categorise_reaction_terms"))

True


In [ ]:
import importlib
importlib.reload(pv_analysis)

categorised_reactions = pv_analysis.categorise_reaction_terms(
    reaction_df
)

print(categorised_reactions["reaction_category"].value_counts())

reaction_category
Other / unclassified                         1800
General symptoms                              434
Clinical conditions / laboratory findings      73
Treatment effectiveness                        39
Injury / physical events                       37
Patient outcomes                               34
Medication use / administration                32
Hypersensitivity / allergic reactions          11
Name: count, dtype: int64


In [ ]:
categorised_reactions[
    categorised_reactions["reaction_category"] != "Other clinical events"
][["reaction_term", "reaction_category"]].drop_duplicates().sort_values(
    "reaction_category"
)

,reaction_term,reaction_category
155,Haemoglobin decreased,Clinical conditions / laboratory findings
853,Type 2 diabetes mellitus,Clinical conditions / laboratory findings
20,Pneumonia,Clinical conditions / laboratory findings
7,Renal impairment,Clinical conditions / laboratory findings
115,Anaemia,Clinical conditions / laboratory findings
...,...,...
625,Muscle spasms,Other / unclassified
627,White blood cell count decreased,Other / unclassified
610,Gastric haemorrhage,Other / unclassified
25,Death,Patient outcomes


In [ ]:
other_terms = (
    categorised_reactions.loc[
        categorised_reactions["reaction_category"] == "Other clinical events",
        "reaction_term"
    ]
    .value_counts()
    .head(30)
)

print(other_terms)

Series([], Name: count, dtype: int64)


In [ ]:
import importlib
importlib.reload(pv_analysis)

categorised_reactions = pv_analysis.categorise_reaction_terms(
    reaction_df
)

print(categorised_reactions["reaction_category"].value_counts())

reaction_category
Other / unclassified                         1800
General symptoms                              434
Clinical conditions / laboratory findings      73
Treatment effectiveness                        39
Injury / physical events                       37
Patient outcomes                               34
Medication use / administration                32
Hypersensitivity / allergic reactions          11
Name: count, dtype: int64


In [ ]:
print("Total reaction records:", len(categorised_reactions))

print(
    "Unique reaction terms:",
    categorised_reactions["reaction_term"].nunique()
)

Total reaction records: 2460
Unique reaction terms: 855


In [ ]:
print("Total reaction records:", len(categorised_reactions))
print("Unique reaction terms:", categorised_reactions["reaction_term"].nunique())
print("\nCategory distribution:")
print(categorised_reactions["reaction_category"].value_counts())

Total reaction records: 2460
Unique reaction terms: 855

Category distribution:
reaction_category
Other / unclassified                         1800
General symptoms                              434
Clinical conditions / laboratory findings      73
Treatment effectiveness                        39
Injury / physical events                       37
Patient outcomes                               34
Medication use / administration                32
Hypersensitivity / allergic reactions          11
Name: count, dtype: int64


In [ ]:
unclassified = categorised_reactions[
    categorised_reactions["reaction_category"] == "Other / unclassified"
]

print("Unclassified reaction records:", len(unclassified))
print(
    "Unclassified unique terms:",
    unclassified["reaction_term"].nunique()
)

Unclassified reaction records: 1800
Unclassified unique terms: 822


In [ ]:
print(categorised_reactions["reaction_category"].value_counts())

reaction_category
Other / unclassified                         1800
General symptoms                              434
Clinical conditions / laboratory findings      73
Treatment effectiveness                        39
Injury / physical events                       37
Patient outcomes                               34
Medication use / administration                32
Hypersensitivity / allergic reactions          11
Name: count, dtype: int64


In [ ]:
folder = "/content/drive/MyDrive/PV_project"

categorised_reactions.to_csv(
    f"{folder}/pv_reactions_categorised.csv",
    index=False
)

pv_review_df.to_csv(
    f"{folder}/pv_reports_review.csv",
    index=False
)

print("Dashboard datasets saved successfully.")

Dashboard datasets saved successfully.


In [1]:
from pathlib import Path

project_folder = Path("/content/drive/MyDrive/PV_project")

for filename in [
    "app.py",
    "pv_reports_review.csv",
    "pv_reactions_categorised.csv"
]:
    file_path = project_folder / filename
    print(filename, "exists:", file_path.exists())

app.py exists: True
pv_reports_review.csv exists: True
pv_reactions_categorised.csv exists: True


In [3]:
import pandas as pd

reports = pd.read_csv(
    project_folder / "pv_reports_review.csv"
)

reactions = pd.read_csv(
    project_folder / "pv_reactions_categorised.csv"
)

print("Reports:", reports.shape)
print("Reactions:", reactions.shape)

Reports: (900, 16)
Reactions: (2460, 4)


In [4]:
from pathlib import Path
import pandas as pd

folder = Path("/content/drive/MyDrive/PV_project")

app_path = folder / "app.py"
reports_path = folder / "pv_reports_review.csv"
reactions_path = folder / "pv_reactions_categorised.csv"

# Check that all three files exist
for path in [app_path, reports_path, reactions_path]:
    print(path.name, "exists:", path.exists())

# Check Python syntax
compile(app_path.read_text(), str(app_path), "exec")
print("\nApp syntax check passed.")

# Check dashboard datasets
reports = pd.read_csv(reports_path)
reactions = pd.read_csv(reactions_path)

print("\nReports:", reports.shape)
print("Reactions:", reactions.shape)

print("\nRequired report columns:")
print(
    set([
        "missing_key_data",
        "invalid_serious_code",
        "selected_event_term",
        "review_required",
        "serious"
    ]).issubset(reports.columns)
)

print("\nRequired reaction columns:")
print(
    set([
        "reaction_term",
        "reaction_category"
    ]).issubset(reactions.columns)
)

app.py exists: True
pv_reports_review.csv exists: True
pv_reactions_categorised.csv exists: True

App syntax check passed.

Reports: (900, 16)
Reactions: (2460, 4)

Required report columns:
True

Required reaction columns:
True


In [5]:
!pip -q install streamlit

In [6]:
%cd /content/drive/MyDrive/PV_project
!streamlit run app.py &>/content/drive/MyDrive/PV_project/streamlit.log &

/content/drive/MyDrive/PV_project


In [7]:
!cat /content/drive/MyDrive/PV_project/streamlit.log



2026-11-09 10:28:27.665 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.58.183.44:8501



In [8]:
!pip -q install pyngrok

In [9]:
from pyngrok import ngrok

ngrok.set_auth_token("3LrluoD8Fs1iTr13K2GNTVF25J3_313sJqCnLeRKk2U35cvjw")

public_url = ngrok.connect(8501)
print("Dashboard URL:", public_url)

Dashboard URL: NgrokTunnel: "https://thesaurus-curdle-handsaw.ngrok-free.dev" -> "http://localhost:8501"


In [10]:
from pathlib import Path

folder = Path("/content/drive/MyDrive/PV_project")

for filename in [
    "app.py",
    "pv_analysis.py",
    "pv_reports_review.csv",
    "pv_reactions_categorised.csv"
]:
    file_path = folder / filename
    size_kb = file_path.stat().st_size / 1024
    print(f"{filename}: {size_kb:.1f} KB")

app.py: 7.8 KB
pv_analysis.py: 3.9 KB
pv_reports_review.csv: 232.8 KB
pv_reactions_categorised.csv: 115.8 KB


In [11]:
from pathlib import Path

folder = Path("/content/drive/MyDrive/PV_project")

requirements = """streamlit
pandas
requests
"""

(folder / "requirements.txt").write_text(requirements)

print("requirements.txt created successfully.")

requirements.txt created successfully.


In [12]:
from pathlib import Path
from getpass import getpass
import requests

# Repository details
owner = "TegaBusola"
repo = "Pharmacovigilance-data-quality-analytics"
folder = Path("/content/drive/MyDrive/PV_project")

# Files to upload
files = [
    "app.py",
    "pv_analysis.py",
    "pv_reports_review.csv",
    "pv_reactions_categorised.csv",
    "requirements.txt",
    "01_data_preparation.ipynb",
    "02_reaction_categorisation.ipynb"
]

# Check all files exist before uploading
missing = [name for name in files if not (folder / name).exists()]

if missing:
    print("Missing files:", missing)
else:
    print("All project files are present.")

Missing files: ['01_data_preparation.ipynb', '02_reaction_categorisation.ipynb']


In [18]:
from pathlib import Path

project_folder = Path("/content/drive/MyDrive/PV_project")
notebook_folder = Path("/content/drive/MyDrive/Colab Notebooks/PV_project")

project_files = [
    "app.py",
    "pv_analysis.py",
    "pv_reports_review.csv",
    "pv_reactions_categorised.csv",
    "requirements.txt"
]

notebook_files = [
    "01_data_preparation.ipynb",
    "02_reaction_categorisation.ipynb"
]

# Check project files
for name in project_files:
    path = project_folder / name
    print(name, "exists:", path.exists())

# Check notebooks
for name in notebook_files:
    path = notebook_folder / name
    print(name, "exists:", path.exists())

app.py exists: True
pv_analysis.py exists: True
pv_reports_review.csv exists: True
pv_reactions_categorised.csv exists: True
requirements.txt exists: True
01_data_preparation.ipynb exists: True
02_reaction_categorisation.ipynb exists: True


In [ ]:
from pathlib import Path
from getpass import getpass
import base64
import requests

owner = "TegaBusola"
repo = "Pharmacovigilance-data-quality-analytics"

project_folder = Path("/content/drive/MyDrive/PV_project")
notebook_folder = Path("/content/drive/MyDrive/Colab Notebooks/PV_project")

token = getpass("Paste your GitHub token (input hidden): ")

headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28"
}

files_to_upload = {
    "app.py": project_folder / "app.py",
    "pv_analysis.py": project_folder / "pv_analysis.py",
    "pv_reports_review.csv": project_folder / "pv_reports_review.csv",
    "pv_reactions_categorised.csv": project_folder / "pv_reactions_categorised.csv",
    "requirements.txt": project_folder / "requirements.txt",
    "notebooks/01_data_preparation.ipynb": notebook_folder / "01_data_preparation.ipynb",
    "notebooks/02_reaction_categorisation.ipynb": notebook_folder / "02_reaction_categorisation.ipynb"
}

for github_path, local_path in files_to_upload.items():
    content = base64.b64encode(local_path.read_bytes()).decode("utf-8")

    url = f"https://api.github.com/repos/{owner}/{repo}/contents/{github_path}"

    response = requests.put(
        url,
        headers=headers,
        json={
            "message": f"Add {github_path}",
            "content": content
        },
        timeout=30
    )

    if response.status_code == 201:
        print(f"Uploaded: {github_path}")
    else:
        print(f"Failed: {github_path} — {response.status_code}")
        print(response.text[:500])

token = None